# train_pairs

Trains the 20 paired het/hom networks on 4-class SHD (64 recurrent neurons, 50 epochs, Adam, lr 2e-3, weight decay 0). Checkpoints land in checkpoints/final_40, per-epoch tau trajectories in checkpoints/tau_trajectories.

## config

In [ ]:
from pathlib import Path
import sys, time, json
import numpy as np
import torch

BASE = Path("..")
# from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/Clean Repo")

sys.path.insert(0, str((BASE / "code").resolve()))
import train_common as tc
import common

DEVICE = tc.DEVICE
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

TASK_KEY = "4class"
MEM_FAMILY = "lognormal"
SEEDS = [101, 202, 210, 340, 440, 550, 660, 710, 820, 930,
         1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]

tc.BASE_PRMS["nb_recurrent"] = 64
tc.BASE_PRMS["batch_size"] = 256
tc.BASE_PRMS["nb_epochs"] = 50
tc.BASE_PRMS["early_stop_patience"] = 10
tc.BASE_PRMS["optimizer"] = "adam"
tc.BASE_PRMS["weight_decay"] = 0.0    # build_seeded_pair also forces 0.0 on both arms
tc.BASE_PRMS["lr"] = 2e-3
tc.BASE_PRMS["lr_ab"] = 2e-3
tc.BASE_PRMS["lr_patience"] = 5
tc.BASE_PRMS["lr_factor"] = 0.5
tc.BASE_PRMS["lr_min"] = 5e-5
tc.BASE_PRMS["compile_model"] = False

CKPT_OUT = BASE / "checkpoints" / "final_40"
TRAJ_OUT = BASE / "checkpoints" / "tau_trajectories"
CKPT_OUT.mkdir(parents=True, exist_ok=True)
TRAJ_OUT.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE, "| seeds:", len(SEEDS), "| output:", CKPT_OUT.resolve())


## data

In [ ]:
train_cache, test_cache = tc.load_default_caches()

## train

In [ ]:
def tau_snapshot(model):
    beta = model.network[0].beta.detach().cpu().numpy().ravel()
    return [float(x) for x in tc.decay_to_tau_ms(beta, 1e-3)]

def make_epoch_cb(store):
    def _cb(epoch, model, history):
        store.append({"epoch": int(epoch),
                      "test_acc": float(history["test_acc"][-1]),
                      "tau_mem_ms": tau_snapshot(model)})
    return _cb

results = []
for seed in SEEDS:
    het_path = CKPT_OUT / f"{common.net_name(seed, 'het')}.pt"
    hom_path = CKPT_OUT / f"{common.net_name(seed, 'hom')}.pt"
    if het_path.exists() and hom_path.exists():
        print(f"seed {seed}: both exist, skip")
        continue

    pair = tc.build_seeded_pair(master_seed=seed, task_key=TASK_KEY, mem_distribution_family=MEM_FAMILY)
    meta = pair["metadata"]
    hp, op = pair["hetero_prms"], pair["hom_prms"]
    assert hp["het_ab"] == 1 and hp["train_ab"] == 1
    assert op["train_hom_ab"] == 1
    assert hp["weight_decay"] == 0.0 and op["weight_decay"] == 0.0
    assert meta["linear_sync_verified"], meta["linear_sync_mismatched"]

    plan = [("hom", pair["hom_model"], op, hom_path, meta["hom_hidden_tau_mem_ms"]),
            ("het", pair["hetero_model"], hp, het_path, meta["hetero_hidden_tau_mem_ms"])]
    for role, model, prms, out_path, init_tau in plan:
        if out_path.exists():
            print(f"seed {seed} {role}: exists, skip")
            continue
        print(f"\nseed {seed} :: {role}")
        traj = [{"epoch": 0, "test_acc": float("nan"), "tau_mem_ms": tau_snapshot(model)}]
        t0 = time.time()
        history = tc.train_experiment(model, prms, train_cache, test_cache,
                                      epoch_callback=make_epoch_cb(traj))
        elapsed = time.time() - t0
        tc.save_model_checkpoint(
            out_path, model, history, elapsed, prms,
            extra={"label": out_path.stem, "task": tc.TASKS[TASK_KEY]["task_name"],
                   "task_key": TASK_KEY, "pair_seed": seed,
                   "pair_role": ("heterogeneous_sampled" if role == "het" else "homogeneous_anchor"),
                   "run_label": "final_40", "mem_distribution_family": MEM_FAMILY,
                   "hidden_tau_syn_ms": meta["hidden_tau_syn_ms"],
                   "hidden_tau_mem_ms": init_tau,
                   "hidden_tau_mem_geom_mean_ms": meta["hidden_tau_mem_geom_mean_ms"],
                   "paired_hetero_hidden_tau_mem_ms": meta["hetero_hidden_tau_mem_ms"],
                   "output_tau_syn_ms": meta["output_tau_syn_ms"],
                   "output_tau_mem_ms": meta["output_tau_mem_ms"],
                   "linear_sync_verified": meta["linear_sync_verified"],
                   "linear_sync_checked": meta["linear_sync_checked"],
                   "tau_is_learned": True, "tau_final_mem_ms": traj[-1]["tau_mem_ms"]})
        with open(TRAJ_OUT / f"4class_seed{seed}_{role}_tautraj.json", "w") as fh:
            json.dump({"seed": seed, "role": role, "epochs": traj}, fh)
        results.append((seed, role, float(history["test_acc"][-1])))
        print(f"saved {out_path.name} acc={history['test_acc'][-1]:.3f} ({elapsed/60:.1f} min)")

for seed, role, acc in results:
    print(f"seed {seed} {role}: {acc:.3f}")